# Sprint 01 — HOS Core Library Validation

This notebook contains two falsifiable unit tests that decide whether the HOS core (`signal_utils.py`) is trustworthy, plus a sparsity contract test for the trispectrum.

---

## Methods — Theoretical Basis

### Bispectrum and Trispectrum Definitions

The bispectrum $B_{xxx}(f_l, f_m)$ is the third-order cross-spectrum defined as the expectation of the triple product of Fourier coefficients:

$$B_{xxx}(f_l, f_m) = \mathbb{E}\!\left[X(f_l)\, X(f_m)\, X^*(f_l+f_m)\right], \quad l + m \le N \tag{Sinha 2007, Eq. 2}$$

where $X(f)$ denotes the discrete Fourier transform of the probe signal and the asterisk denotes complex conjugation. The fourth-order counterpart, the trispectrum, extends this to three independent frequency arguments:

$$T_{xxxx}(f_l, f_m, f_n) = \mathbb{E}\!\left[X^*(f_l)\, X^*(f_m)\, X^*(f_n)\, X(f_l+f_m+f_n)\right] \tag{Sinha 2007, Eq. 3}$$

Sinha (2007) applied both estimators to distinguish a breathing transverse crack from parallel coupling misalignment in a rotating shaft, demonstrating that the two fault types produce qualitatively different bispectral and trispectral topologies even when their frequency-domain power spectra are similar. The estimator settings used by Sinha—sampling frequency $f_s = 2560$ Hz, frequency resolution $\Delta f = 1.25$ Hz, 50 segments at 50% overlap—are reproduced here (Sinha, 2007, §3.3).

### Kim–Powers Normalization

The raw bispectrum is difficult to interpret directly because its magnitude depends on the signal amplitude. Kim & Powers (1979) introduced the squared bicoherence $b^2$, which normalizes the bispectrum to the interval $[0, 1]$:

$$b^2(f_l, f_m) = \frac{\left|\langle X(f_l)\, X(f_m)\, X^*(f_l+f_m) \rangle\right|^2}{\langle |X(f_l)\, X(f_m)|^2 \rangle \cdot \langle |X(f_l+f_m)|^2 \rangle}$$

where angle brackets denote averaging over $K$ non-overlapping (or overlap-corrected) segments. The bound $b^2 \in [0, 1]$ follows directly from the Cauchy–Schwarz inequality (Kim & Powers, 1979). A value near unity at $(f_l, f_m)$ indicates that the phases of $X(f_l)$, $X(f_m)$, and $X(f_l+f_m)$ are statistically locked across records—the signature of quadratic phase coupling (QPC). Independent harmonic components whose phases vary randomly between records yield $b^2 \approx 0$ in the limit of large $K$.

### Direct Estimator and Non-Redundant Region

The computational framework follows the direct (segment-averaging) method formalized by Collis, White & Hammond (1998): the signal is divided into $K$ windowed segments, an FFT is computed once per segment, and the triple product is accumulated and averaged. The non-redundant principal domain—restricted to $f_l \le f_m$ and $f_l + f_m \le f_{Nyquist}$ for the bispectrum, and analogously for the trispectrum—is the only region that is computed and stored; all other entries are set to zero by symmetry (Collis et al., 1998).

### References

Kim, Y. C., & Powers, E. J. (1979). Digital bispectral analysis and its applications to nonlinear wave interactions. *IEEE Transactions on Plasma Science*, *7*(2), 120–131.

Sinha, J. K. (2007). Higher order spectra for crack and misalignment identification in the shaft of a rotating machine. *Structural Health Monitoring*, *6*(4), 325–334.

Collis, W. B., White, P. R., & Hammond, J. K. (1998). Higher-order spectra: the bispectrum and trispectrum. *Mechanical Systems and Signal Processing*, *12*(3), 375–394.

## Setup

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for notebook runs without display
import matplotlib.pyplot as plt

from signal_utils import bispectrum, bicoherence, trispectrum
from plot_utils import plot_bispectrum_surface, plot_trispectrum_balls

fs = 2560.0
T_len = 25.0
t = np.arange(0, T_len, 1 / fs)

f = 12.5    # Hz — close to Sinha's 1X at 750 RPM
phi = 1.3   # arbitrary fixed phase

print(f"Signal length: {len(t)} samples  ({T_len} s at {fs} Hz)")
print(f"Fundamental: {f} Hz  (1X = {f*60:.0f} RPM)")

---
## Test A — Positive: enforced quadratic phase coupling

A signal with three harmonically related components whose phases are locked across the entire record must yield $b^2 \approx 1$ at the coupled frequency pairs.

In [ ]:
x = (
    np.cos(2 * np.pi * f * t)
    + 0.5 * np.cos(2 * np.pi * 2 * f * t + 2 * phi)   # 2f — quadratically coupled
    + 0.5 * np.cos(2 * np.pi * 3 * f * t + 3 * phi)   # 3f — cubically coupled
)

B, b2, freqs = bispectrum(x, fs)

i_f  = int(np.argmin(np.abs(freqs - f)))
i_2f = int(np.argmin(np.abs(freqs - 2 * f)))

print(f"Frequency resolution: {freqs[1] - freqs[0]:.4f} Hz  (target 1.25 Hz)")
print(f"Number of segments  : {len(x) // (len(freqs) - 1)}  (target 50 at 50% overlap)")
print()
print(f"b²(f, f)   = {b2[i_f,  i_f]:.4f}  (assert ≥ 0.95)")
print(f"b²(f, 2f)  = {b2[i_f, i_2f]:.4f}  (assert ≥ 0.95)")

assert b2[i_f,  i_f]  >= 0.95, f"b²(f,f)  = {b2[i_f,i_f]:.3f}, want ≥ 0.95"
assert b2[i_f, i_2f] >= 0.95, f"b²(f,2f) = {b2[i_f,i_2f]:.3f}, want ≥ 0.95"
print("\nTest A PASSED ✓")

In [ ]:
fig = plot_bispectrum_surface(B, freqs, fmax_hz=50.0)
fig.suptitle("Test A — Enforced QPC: bispectrum surface (compare Sinha Fig. 5)")
plt.show()

---
## Test B — Negative: independent random phases per segment

The same three harmonics but with phases re-randomized each second break the phase coupling between segments.  A correct bicoherence estimator must return $b^2 \approx 0$ everywhere.  A common implementation bug—normalizing the denominator **per segment** rather than averaging it first—produces $b^2 \approx 1$ even for this uncoupled signal.

In [ ]:
# Phase randomization must align with the bispectrum-segment granularity, not
# a coarser timescale.  Using blocks of exactly nfft samples with noverlap=0
# ensures every segment sees independent random phases, driving b² → 0.
rng = np.random.default_rng(42)
nfft_b = int(round(fs / 1.25))   # = 2048, same nfft used by bispectrum default
t_block = np.arange(nfft_b) / fs
n_blocks = 50   # one block per bispectrum segment

x_uncoupled = np.concatenate([
    (
        np.cos(2 * np.pi * f * t_block + rng.uniform(0, 2 * np.pi))
        + 0.5 * np.cos(2 * np.pi * 2 * f * t_block + rng.uniform(0, 2 * np.pi))
        + 0.5 * np.cos(2 * np.pi * 3 * f * t_block + rng.uniform(0, 2 * np.pi))
    )
    for _ in range(n_blocks)
])

# noverlap=0 aligns segments with blocks — no segment spans two phase-independent blocks
_, b2_neg, _ = bispectrum(x_uncoupled, fs, nfft=nfft_b, noverlap=0)

print(f"Segments used: {n_blocks}")
print(f"max b² (uncoupled) = {b2_neg.max():.4f}  (assert ≤ 0.20)")

assert b2_neg.max() <= 0.20, f"max b² = {b2_neg.max():.3f}, want ≤ 0.20"
print("Test B PASSED ✓")

---
## Test C — Trispectrum sparsity contract

For the enforced-QPC signal from Test A, the only non-redundant trispectrum peak above the 0.1 threshold must be at $(f, f, f)$.  The dict must contain $\le 10$ entries, confirming that the sparse representation does not fill up with spurious peaks.

In [ ]:
T_dict, freqs_t = trispectrum(x, fs, threshold=0.10)

print(f"Trispectrum entries above 0.10 threshold: {len(T_dict)}  (assert ≤ 10)")
print(f"Keys: {list(T_dict.keys())}")

T_abs = {k: abs(v) for k, v in T_dict.items()}
T_max = max(T_abs.values())
norm_at_peak = T_abs.get((i_f, i_f, i_f), 0.0) / T_max
print(f"|T(f,f,f)| / max|T| = {norm_at_peak:.4f}  (assert ≥ 0.95)")

assert (i_f, i_f, i_f) in T_dict, f"(i_f, i_f, i_f) = ({i_f},{i_f},{i_f}) not in T_dict"
assert norm_at_peak >= 0.95, f"|T(f,f,f)|/max = {norm_at_peak:.3f}, want ≥ 0.95"
assert len(T_dict) <= 10, f"T_dict has {len(T_dict)} entries, want ≤ 10"
print("\nTest C PASSED ✓")

In [ ]:
fig = plot_trispectrum_balls(T_dict, freqs_t, fmax_hz=50.0, amp_min=0.10)
fig.suptitle("Test C — Enforced QPC: trispectrum balls (compare Sinha Fig. 7)")
plt.show()

---
## Summary

| Test | What it checks | Criterion |
|------|---------------|----------|
| A | QPC signal → high bicoherence at (f,f) and (f,2f) | b² ≥ 0.95 |
| B | Uncoupled phases → low bicoherence everywhere | max b² ≤ 0.20 |
| C | Sparse trispectrum: peak at (f,f,f), ≤ 10 entries | |T(f,f,f)|/max ≥ 0.95 and len ≤ 10 |

All three tests passing confirms that `signal_utils.py` correctly implements the Kim–Powers normalized bispectrum/trispectrum and that the bicoherence denominator is averaged across segments before division (not per-segment, which is the common bug that Test B detects).